In [3]:
import requests
from pathlib import Path
# 香港入境事务处官方 CSV
url = (    
"https://www.immd.gov.hk/opendata/eng/"    
"transport/immigration_clearance/"    
"statistics_on_daily_passenger_traffic.csv"
)
# 下载最新数据
response = requests.get(url, timeout=60)
response.raise_for_status()
# 保存原始数据
folder = Path("data/raw")
folder.mkdir(parents=True, exist_ok=True)
file_path = folder / "immd_daily.csv"
file_path.write_bytes(response.content)
print("Download completed!")
print("Saved to:", file_path)

Download completed!
Saved to: data\raw\immd_daily.csv


In [4]:
import pandas as pd
df = pd.read_csv("data/raw/immd_daily.csv")
print(df.head())
print(df.columns.tolist())
print(df.shape)

         Date                   Control Point Arrival / Departure  \
0  01-01-2021                         Airport             Arrival   
1  01-01-2021                         Airport           Departure   
2  01-01-2021  Express Rail Link West Kowloon             Arrival   
3  01-01-2021  Express Rail Link West Kowloon           Departure   
4  01-01-2021                        Hung Hom             Arrival   

   Hong Kong Residents  Mainland Visitors  Other Visitors  Total  Unnamed: 7  
0                  341                  0               9    350         NaN  
1                  803                 17              28    848         NaN  
2                    0                  0               0      0         NaN  
3                    0                  0               0      0         NaN  
4                    0                  0               0      0         NaN  
['Date', 'Control Point', 'Arrival / Departure', 'Hong Kong Residents', 'Mainland Visitors', 'Other Visitors', 

In [5]:
# 查看所有口岸名称
print("所有口岸：")
print(df["Control Point"].unique())
# 查看出入境方向
print("\n出入境方向：")
print(df["Arrival / Departure"].unique())
# 查看日期范围
print("\n最早日期：")
print(df["Date"].iloc[0])
print("\n最后日期：")
print(df["Date"].iloc[-1])

所有口岸：
<StringArray>
[                       'Airport', 'Express Rail Link West Kowloon',
                       'Hung Hom',                          'Lo Wu',
          'Lok Ma Chau Spur Line',                 'Heung Yuen Wai',
  'Hong Kong-Zhuhai-Macao Bridge',                    'Lok Ma Chau',
                     'Man Kam To',                    'Sha Tau Kok',
                   'Shenzhen Bay',           'China Ferry Terminal',
                'Harbour Control',        'Kai Tak Cruise Terminal',
           'Macau Ferry Terminal',        'Tuen Mun Ferry Terminal',
           'Macao Ferry Terminal',                      'Huanggang']
Length: 18, dtype: str

出入境方向：
<StringArray>
['Arrival', 'Departure']
Length: 2, dtype: str

最早日期：
01-01-2021

最后日期：
31-10-2026


In [6]:
# 1. 将日期转换成真正的日期格式
df["Date"] = pd.to_datetime(    
df["Date"],    
format="%d-%m-%Y"
)
# 2. 设置研究时间
start_date = "2023-02-06"
end_date = "2026-10-31"
# 3. 定义北上口岸
northbound_ports = [    
"Lo Wu",    
"Lok Ma Chau Spur Line",    
"Lok Ma Chau",    
"Shenzhen Bay",    
"Heung Yuen Wai",    
"Hong Kong-Zhuhai-Macao Bridge",    
"Express Rail Link West Kowloon"
]
# 4. 筛选日期、口岸和出境方向
northbound = df[
    (df["Date"] >= start_date) &
    (df["Date"] <= end_date) &
    (df["Control Point"].isin(northbound_ports)) &
    (df["Arrival / Departure"] == "Departure")
].copy()
# 5. 检查筛选结果
print("筛选后的前5行：")
print(northbound.head())
print("\n筛选后的数据量：")
print(northbound.shape)
print("\n包含的口岸：")
print(northbound["Control Point"].unique())

筛选后的前5行：
            Date                   Control Point Arrival / Departure  \
23789 2023-02-06  Express Rail Link West Kowloon           Departure   
23793 2023-02-06                           Lo Wu           Departure   
23795 2023-02-06           Lok Ma Chau Spur Line           Departure   
23797 2023-02-06                  Heung Yuen Wai           Departure   
23799 2023-02-06   Hong Kong-Zhuhai-Macao Bridge           Departure   

       Hong Kong Residents  Mainland Visitors  Other Visitors  Total  \
23789                 7532               3837             378  11747   
23793                29686               3828             142  33656   
23795                23312               4419             155  27886   
23797                 2813                363               6   3182   
23799                12584                901            2310  15795   

       Unnamed: 7  
23789         NaN  
23793         NaN  
23795         NaN  
23797         NaN  
23799         NaN  

筛选后的

In [7]:
# 检查筛选后的数据
print("筛选后的数据行数：")
print(len(northbound))
print("\n实际包含的口岸：")
print(northbound["Control Point"].unique())
print("\n各口岸的数据行数：")
print(northbound["Control Point"].value_counts())
print("\n研究期间覆盖的天数：")
print(northbound["Date"].nunique())

筛选后的数据行数：
9529

实际包含的口岸：
<StringArray>
['Express Rail Link West Kowloon',                          'Lo Wu',
          'Lok Ma Chau Spur Line',                 'Heung Yuen Wai',
  'Hong Kong-Zhuhai-Macao Bridge',                    'Lok Ma Chau',
                   'Shenzhen Bay']
Length: 7, dtype: str

各口岸的数据行数：
Control Point
Express Rail Link West Kowloon    1364
Lo Wu                             1364
Lok Ma Chau Spur Line             1364
Heung Yuen Wai                    1364
Hong Kong-Zhuhai-Macao Bridge     1364
Shenzhen Bay                      1364
Lok Ma Chau                       1345
Name: count, dtype: int64

研究期间覆盖的天数：
1364


In [8]:
# Y1：每日香港居民北上人次
daily_northbound = (    
northbound.groupby("Date")["Hong Kong Residents"]    
.sum()    
.reset_index()
)
# 修改列名
daily_northbound.columns = [    
"Date",    
"Northbound_Passengers"
]
# 查看前10天
print("每日港人北上人次：")
print(daily_northbound.head(10))
# 检查总天数
print("\n数据总天数：")
print(len(daily_northbound))
# 计算每日平均人数
print("\n每日平均北上人次：")
print(daily_northbound["Northbound_Passengers"].mean())

每日港人北上人次：
        Date  Northbound_Passengers
0 2023-02-06                  91029
1 2023-02-07                  84887
2 2023-02-08                  88115
3 2023-02-09                  93738
4 2023-02-10                 133457
5 2023-02-11                 168102
6 2023-02-12                 111750
7 2023-02-13                 106066
8 2023-02-14                 101605
9 2023-02-15                 102414

数据总天数：
1364

每日平均北上人次：
251783.53225806452


In [9]:
from pathlib import Path
# 创建输出文件夹
output_folder = Path("data/processed")
output_folder.mkdir(parents=True, exist_ok=True)
# 保存 Y1
daily_northbound.to_csv(    
output_folder / "daily_northbound.csv",    
index=False,    
encoding="utf-8-sig"
)
print("Y1 数据保存成功！")

Y1 数据保存成功！


In [10]:
# Y2：计算每日各口岸的客流份额
# 1. 按日期和口岸汇总香港居民出境人数
port_daily = (    
northbound.groupby(["Date", "Control Point"])[        
"Hong Kong Residents"
    ]    
.sum()    
.reset_index()
)
# 2. 合并Y1每日总人数
port_daily = port_daily.merge(    
daily_northbound,    
on="Date",    
how="left"
)
# 3. 计算各口岸份额
port_daily["Port_Share"] = (    
port_daily["Hong Kong Residents"] /    
port_daily["Northbound_Passengers"]
)
# 4. 查看结果
print("Y2：每日各口岸客流份额")
print(port_daily.head(15))
print("\n总行数：")
print(len(port_daily))

Y2：每日各口岸客流份额
         Date                   Control Point  Hong Kong Residents  \
0  2023-02-06  Express Rail Link West Kowloon                 7532   
1  2023-02-06                  Heung Yuen Wai                 2813   
2  2023-02-06   Hong Kong-Zhuhai-Macao Bridge                12584   
3  2023-02-06                           Lo Wu                29686   
4  2023-02-06                     Lok Ma Chau                 3899   
5  2023-02-06           Lok Ma Chau Spur Line                23312   
6  2023-02-06                    Shenzhen Bay                11203   
7  2023-02-07  Express Rail Link West Kowloon                 6111   
8  2023-02-07                  Heung Yuen Wai                 3316   
9  2023-02-07   Hong Kong-Zhuhai-Macao Bridge                11821   
10 2023-02-07                           Lo Wu                28787   
11 2023-02-07                     Lok Ma Chau                 3404   
12 2023-02-07           Lok Ma Chau Spur Line                21572   
13 2023

In [13]:
# 检查每日7个口岸份额之和
share_check = (    
port_daily.groupby("Date")["Port_Share"]    
.sum()
)
print("前10天的份额合计：")
print(share_check.head(10))
# 检查是否接近100%
import numpy as np
print("\n所有日期份额是否等于100%：")
print(np.allclose(share_check, 1))

前10天的份额合计：
Date
2023-02-06    1.0
2023-02-07    1.0
2023-02-08    1.0
2023-02-09    1.0
2023-02-10    1.0
2023-02-11    1.0
2023-02-12    1.0
2023-02-13    1.0
2023-02-14    1.0
2023-02-15    1.0
Name: Port_Share, dtype: float64

所有日期份额是否等于100%：
True


In [14]:
# 保存各口岸每日份额
port_daily.to_csv(    
output_folder / "port_daily.csv",    
index=False,    
encoding="utf-8-sig"
)
print("Y2 数据保存成功！")

Y2 数据保存成功！


In [15]:
# Y3：每日内地访客南下人次
# 1. 筛选研究日期和7个口岸
mainland_arrival = df[
    (df["Date"] >= "2023-02-06") &
    (df["Date"] <= "2026-10-31") &
    (df["Control Point"].isin(northbound_ports)) &
    (df["Arrival / Departure"] == "Arrival")
].copy()
# 2. 按日期汇总内地访客入境人数
daily_mainland = (    
mainland_arrival.groupby("Date")["Mainland Visitors"]    
.sum()    
.reset_index()
)
# 3. 修改列名
daily_mainland.columns = [    
"Date",    
"Mainland_Arrivals"
]
# 4. 查看结果
print("Y3：每日内地访客南下人次")
print(daily_mainland.head(10))
print("\n数据总天数：")
print(len(daily_mainland))
print("\n每日平均南下人次：")
print(daily_mainland["Mainland_Arrivals"].mean())

Y3：每日内地访客南下人次
        Date  Mainland_Arrivals
0 2023-02-06              17754
1 2023-02-07              18914
2 2023-02-08              21853
3 2023-02-09              23258
4 2023-02-10              32859
5 2023-02-11              43245
6 2023-02-12              36188
7 2023-02-13              31435
8 2023-02-14              29351
9 2023-02-15              33659

数据总天数：
1364

每日平均南下人次：
85269.57184750734


In [16]:
# Y4：每日香港居民净出境人次
# 1. 筛选研究日期和7个口岸
hk_all = df[
    (df["Date"] >= "2023-02-06") &
    (df["Date"] <= "2026-10-31") &
    (df["Control Point"].isin(northbound_ports))
].copy()
# 2. 分别汇总每日出境和入境人次
hk_direction = (    
hk_all.groupby(
        ["Date", "Arrival / Departure"]
    )["Hong Kong Residents"]    
.sum()    
.unstack(fill_value=0)
)
# 3. 计算净出境
hk_direction["Net_Departure"] = (    
hk_direction["Departure"] -    
hk_direction["Arrival"]
)
# 4. 整理结果
daily_net = (    
hk_direction[["Net_Departure"]]    
.reset_index()
)
print("Y4：每日香港居民净出境人次")
print(daily_net.head(10))
print("\n数据总天数：")
print(len(daily_net))

Y4：每日香港居民净出境人次
Arrival / Departure       Date  Net_Departure
0                   2023-02-06           5617
1                   2023-02-07           -891
2                   2023-02-08            -61
3                   2023-02-09           5081
4                   2023-02-10          36305
5                   2023-02-11          58906
6                   2023-02-12         -85752
7                   2023-02-13          -9092
8                   2023-02-14            678
9                   2023-02-15          -4439

数据总天数：
1364


In [17]:
# 合并每日客流数据
master_passenger = (    
daily_northbound    
.merge(daily_mainland, on="Date", how="left")    
.merge(daily_net, on="Date", how="left")
)
# 查看结果
print(master_passenger.head(10))
print("\n总行数和列数：")
print(master_passenger.shape)
# 保存文件
master_passenger.to_csv(    
output_folder / "passenger_daily.csv",    
index=False,    
encoding="utf-8-sig"
)
print("\n每日客流总表保存成功！")

        Date  Northbound_Passengers  Mainland_Arrivals  Net_Departure
0 2023-02-06                  91029              17754           5617
1 2023-02-07                  84887              18914           -891
2 2023-02-08                  88115              21853            -61
3 2023-02-09                  93738              23258           5081
4 2023-02-10                 133457              32859          36305
5 2023-02-11                 168102              43245          58906
6 2023-02-12                 111750              36188         -85752
7 2023-02-13                 106066              31435          -9092
8 2023-02-14                 101605              29351            678
9 2023-02-15                 102414              33659          -4439

总行数和列数：
(1364, 4)

每日客流总表保存成功！


In [18]:
# 检查每日客流总表
print("1. 缺失值：")
print(master_passenger.isna().sum())
print("\n2. 重复日期：")
print(master_passenger["Date"].duplicated().sum())
print("\n3. 各变量的基本统计：")
print(master_passenger.describe())
print("\n4. 是否有负数客流：")
print(
    (master_passenger[
        ["Northbound_Passengers", "Mainland_Arrivals"]
    ] < 0).sum()
)

1. 缺失值：
Date                     0
Northbound_Passengers    0
Mainland_Arrivals        0
Net_Departure            0
dtype: int64

2. 重复日期：
0

3. 各变量的基本统计：
                      Date  Northbound_Passengers  Mainland_Arrivals  \
count                 1364            1364.000000        1364.000000   
mean   2024-12-18 12:00:00          251783.532258       85269.571848   
min    2023-02-06 00:00:00            6649.000000        1744.000000   
25%    2024-01-12 18:00:00          198713.750000       63157.500000   
50%    2024-12-18 12:00:00          243066.000000       76949.500000   
75%    2025-11-24 06:00:00          290525.500000       99585.000000   
max    2026-10-31 00:00:00          636449.000000      256340.000000   
std                    NaN           82814.541895       32707.724327   

       Net_Departure  
count    1364.000000  
mean      254.585044  
min   -457436.000000  
25%    -24444.250000  
50%       956.000000  
75%     52820.000000  
max    384003.000000  
std     8874

In [19]:
# 1. 找出净出境最低的10天
lowest_net = master_passenger.nsmallest(
    10, "Net_Departure"
)

print("净出境最低的10天：")
print(lowest_net.to_string(index=False))

# 2. 找出北上人数最多的10天
highest_northbound = master_passenger.nlargest(
    10, "Northbound_Passengers"
)

print("\n北上人数最多的10天：")
print(highest_northbound.to_string(index=False))

# 3. 再次检查缺失值
print("\n所有列缺失值数量：")
print(master_passenger.isna().sum().to_string())


净出境最低的10天：
      Date  Northbound_Passengers  Mainland_Arrivals  Net_Departure
2026-04-07                 201740              56068        -457436
2026-05-25                 281368              71748        -387236
2025-04-21                 193487              62774        -379210
2026-10-19                 215335              71917        -374113
2024-07-01                 193888              66161        -368277
2023-10-23                 196675              58537        -317799
2026-09-27                 226890              95514        -310864
2024-06-10                 197038              66831        -301615
2024-10-13                 191030              83588        -301546
2026-06-21                 233734              84217        -292902

北上人数最多的10天：
      Date  Northbound_Passengers  Mainland_Arrivals  Net_Departure
2025-04-18                 636449              61375         384003
2026-04-03                 613435              63224         358239
2024-03-29              

In [11]:
# 查看新皇岗及原落马洲口岸的数据

check = df[
    (df["Date"] >= "2026-10-10") &
    (df["Date"] <= "2026-10-15") &
    (df["Control Point"].isin([
        "Lok Ma Chau",
        "Huanggang"
    ]))
].copy()

print(
    check[
        [
            "Date",
            "Control Point",
            "Arrival / Departure",
            "Hong Kong Residents"
        ]
    ].to_string(index=False)
)


      Date Control Point Arrival / Departure  Hong Kong Residents
2026-10-10   Lok Ma Chau             Arrival                17844
2026-10-10   Lok Ma Chau           Departure                19625
2026-10-11   Lok Ma Chau             Arrival                25547
2026-10-11   Lok Ma Chau           Departure                14836
2026-10-12     Huanggang             Arrival                27597
2026-10-12     Huanggang           Departure                30354
2026-10-12   Lok Ma Chau             Arrival                 5726
2026-10-12   Lok Ma Chau           Departure                 2438
2026-10-13     Huanggang             Arrival                33374
2026-10-13     Huanggang           Departure                35451
2026-10-14     Huanggang             Arrival                32480
2026-10-14     Huanggang           Departure                34683
2026-10-15     Huanggang             Arrival                33652
2026-10-15     Huanggang           Departure                36150
